# 0.1 bis · Réseaux OSM marche/vélo consolidés

Ce notebook construit les couches réseau marche et vélo à partir d'OpenStreetMap.

Principes de cette version bis :
- paramètres structurants regroupés en début de notebook ;
- territoire utilisé comme clé de configuration des chemins et de l'AOI ;
- suppression optionnelle des `sidewalk_proxy_road_axis` redondants lorsqu'ils sont déjà couverts par des `walk_dedicated` proches ;
- nettoyage volontairement simple : couverture linéaire dans un buffer métrique, sans fallback ni règles de graphe complexes.


In [1]:

from __future__ import annotations

import json
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")
import math
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
from shapely.ops import unary_union

import osmnx as ox

pd.set_option('display.max_columns', None)  

## 0 · Paramètres

Les cellules suivantes définissent le territoire, les chemins, les options d'extraction et les seuils de nettoyage. Les variables historiques en minuscules sont conservées pour limiter les changements dans le reste du notebook.


In [2]:
# CRS · séparer opérations métriques et export final
OPERATION_CRS = "EPSG:2056"  # CRS métrique pour distances, buffers, segmentation
EXPORT_CRS = "EPSG:4326"     # CRS final pour les exports

TERRITORY = "GG"  # "GG" ou "GE"
NETWORK = "bike"  # conservé pour compatibilité avec les cellules aval

INPUT_ROOT = Path("../../Data/input")
OUTPUT_ROOT = Path("../../Data/output")

TERRITORY_CONFIG = {
    "GG": {
        "network_dir": INPUT_ROOT / "networkGG",
        "aoi_shapefile": INPUT_ROOT / "network_agreg/AGGLO_PERIMETRE_AVEC_LAC-SHP/AGGLO_PERIMETRE_AVEC_LAC.shp",
        "output_step1_path": OUTPUT_ROOT / "GG/step-1",
        "output_step2_path": OUTPUT_ROOT / "GG/step-2",
        "output_step3_path": OUTPUT_ROOT / "GG/step-3",
        "places": [
            "Genève, Switzerland",
            "Vandoeuvres, Switzerland",
            "Cologny, Switzerland",
            "Collonge-Bellerive, Switzerland",
            "Choulex, Switzerland",
        ],
    },
    "GE": {
        "network_dir": INPUT_ROOT / "networkGE",
        "aoi_shapefile": INPUT_ROOT / "network_agreg/CAD_LIMITE_CANTON-SHP/CAD_LIMITE_CANTON_POLYGON.shp",
        "output_step1_path": OUTPUT_ROOT / "GE/step-1",
        "output_step2_path": OUTPUT_ROOT / "GE/step-2",
        "output_step3_path": OUTPUT_ROOT / "GE/step-3",
        "places": [
            "Genève, Switzerland",
            "Vandoeuvres, Switzerland",
            "Cologny, Switzerland",
            "Collonge-Bellerive, Switzerland",
            "Choulex, Switzerland",
        ],
    },
}

if TERRITORY not in TERRITORY_CONFIG:
    raise ValueError(f"Territoire inconnu: {TERRITORY}. Valeurs attendues: {list(TERRITORY_CONFIG)}")

cfg = TERRITORY_CONFIG[TERRITORY]

# Variables historiques conservées pour les cellules suivantes.
operation_crs = OPERATION_CRS
export_crs = EXPORT_CRS
territory = TERRITORY
network = NETWORK
index_col = "bike_index" if network == "bike" else "walk_index"

input_file_path = str(INPUT_ROOT)
output_step1_path = str(cfg["output_step1_path"])
output_step2_path = str(cfg["output_step2_path"])
output_step3_path = str(cfg["output_step3_path"])
shapefile_path = cfg["aoi_shapefile"]
places = cfg["places"]
output_dir = cfg["network_dir"]
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Territoire: {territory}")
print(f"Réseau de référence: {network}")
print(f"Dossier de sortie: {output_dir}")

Territoire: GG
Réseau de référence: bike
Dossier de sortie: ../../Data/input/networkGG


In [3]:
# Zone d'étude
AOI_MODE = "shapefile"  # "shapefile" | "geocode"
area_mode = AOI_MODE

# Exports
export_nodes = True
export_edges = True
export_graphml = True
export_formats = ["parquet", "geojson"]  # "parquet" | "geojson" | "gpkg"

# Configuration graphe vélo
bike_directed_graph = True  # True = graphe orienté respectant le sens de circulation

# Enrichissements automatiques
# Ajoute les attributs de nœuds OSM, la surface vélo et les conflits marche/vélo.
enable_enrichments = True

# Options consolidation marche
add_platform_edges = True
add_crossing_edges = False
add_sidewalk_proxy = True

# Proxy trottoir · règles d'inclusion simples
proxy_include_highways = {
    "residential",
    "living_street",
    "service",
    "unclassified",
    "tertiary",
    "path",
    "track",
}



proxy_maxspeed_kmh = 60  # inclure si maxspeed <= seuil, si maxspeed est parsable
proxy_require_sidewalk_tag = False  # True = inclure proxy seulement si sidewalk=* présent

# Traversées synthétiques
crossing_snap_dist_m = 12.0
crossing_halfwidth_m = 6.0
crossing_min_road_classes = {
    "residential",
    "living_street",
    "service",
    "unclassified",
    "tertiary",
    "secondary",
    "primary",
    "path",
    "track",
}

# Nettoyage des proxies redondants
remove_proxy_near_dedicated = True
proxy_duplicate_class = "sidewalk_proxy_road_axis"
proxy_dedicated_classes = {"walk_dedicated"}
proxy_dedicated_buffer_m = 12.0
proxy_min_covered_len_m = 20.0
proxy_min_coverage_ratio = 0.65
proxy_min_dedicated_len_m = 8.0

# Nettoyage géométrique final
deduplicate_by_geometry = True
simplify_tol_m = 0.0  # 0 = off


# --------------------------------------------------
# Gestion de highway=track pour la marche
# --------------------------------------------------
# "walk_track" : track = chemin piéton, classe distincte "walk_track" (rôle dedicated)
# "proxy"      : ancien comportement (track uniquement via le proxy de trottoir)
# "exclude"    : track retiré du réseau piéton (vélo non affecté)
TRACK_WALK_MODE = "walk_track"

# Colonnes OSM conservées pour pondérer les track plus tard
TRACK_EXTRA_COLS = ["tracktype", "surface", "smoothness"]

if TRACK_WALK_MODE == "exclude":
    proxy_include_highways = proxy_include_highways - {"track"}
    crossing_min_road_classes = crossing_min_road_classes - {"track"}

## Fonctions utilitaires (simples et réutilisables)

In [4]:
def load_aoi(area_mode: str, shapefile_path: Path, places: list[str]) -> gpd.GeoDataFrame:
    """Retourne une GeoDataFrame (EPSG:4326) contenant un polygone d'AOI."""
    if area_mode == "shapefile":
        gdf = gpd.read_file(shapefile_path)
        gdf = gdf.to_crs("EPSG:4326") if gdf.crs is not None else gdf.set_crs("EPSG:4326")
        geom = unary_union(gdf.geometry.values)
        return gpd.GeoDataFrame({"name":["aoi"]}, geometry=[geom], crs="EPSG:4326")

    if area_mode == "geocode":
        polys = []
        for p in places:
            g = ox.geocode_to_gdf(p).to_crs("EPSG:4326")
            polys.append(unary_union(g.geometry.values))
        geom = unary_union(polys)
        return gpd.GeoDataFrame({"name":["aoi"]}, geometry=[geom], crs="EPSG:4326")

    raise ValueError(f"area_mode inconnu: {area_mode}")


def to_crs(gdf: gpd.GeoDataFrame, crs: str) -> gpd.GeoDataFrame:
    """Convertit vers le CRS spécifié en gérant les cas sans CRS défini."""
    if gdf.crs is None:
        return gdf.set_crs("EPSG:4326").to_crs(crs)
    return gdf.to_crs(crs)


def to_metric_crs(gdf: gpd.GeoDataFrame, metric_crs: str) -> gpd.GeoDataFrame:
    """Convertit vers un CRS métrique pour les calculs de distance."""
    return to_crs(gdf, metric_crs)


def to_export_crs(gdf: gpd.GeoDataFrame, export_crs: str) -> gpd.GeoDataFrame:
    """Convertit vers le CRS d'export final."""
    return to_crs(gdf, export_crs)


def explode_lines(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf = gdf[gdf.geometry.notna()].copy()
    return gdf.explode(index_parts=False, ignore_index=True)


def simplify_gdf(gdf: gpd.GeoDataFrame, tol_m: float, metric_crs: str) -> gpd.GeoDataFrame:
    """Simplifie en métrique puis reprojette dans le CRS d'origine."""
    if tol_m and tol_m > 0:
        original_crs = gdf.crs
        gdf_metric = to_metric_crs(gdf, metric_crs)
        gdf_metric = gdf_metric.copy()
        gdf_metric["geometry"] = gdf_metric.geometry.simplify(tol_m, preserve_topology=True)
        return to_crs(gdf_metric, str(original_crs))
    return gdf

import math
import numpy as np
from shapely.geometry import LineString

def mean_bearing(line: LineString) -> float:
    """
    Calcule une orientation moyenne simple d'une LineString (en radians).
    Approche volontairement robuste et peu coûteuse.
    """
    if line is None or line.is_empty:
        return np.nan

    coords = list(line.coords)
    if len(coords) < 2:
        return np.nan

    x0, y0 = coords[0]
    x1, y1 = coords[-1]

    dx = x1 - x0
    dy = y1 - y0

    if dx == 0 and dy == 0:
        return np.nan

    return math.atan2(dy, dx)


def safe_object_columns_for_parquet(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Normalise les colonnes object mixtes (ex. osmid liste/scalar) pour éviter les erreurs parquet."""
    out = gdf.copy()
    for col in out.columns:
        if col == out.geometry.name:
            continue
        if out[col].dtype == "object":
            has_list = out[col].apply(lambda v: isinstance(v, (list, tuple, set))).any()
            if has_list:
                out[col] = out[col].apply(
                    lambda v: None if pd.isna(v) else json.dumps(list(v)) if isinstance(v, (list,tuple,set)) else json.dumps([v])
                )
            else:
                out[col] = out[col].apply(lambda v: None if pd.isna(v) else str(v))
    return out


def export_gdf(gdf: gpd.GeoDataFrame, stem: str, export_formats: list[str], output_dir: Path, export_crs: str):
    """Export multi-formats dans le CRS d'export spécifié."""
    gdf_export = to_export_crs(gdf, export_crs)
    
    if "parquet" in export_formats:
        safe_object_columns_for_parquet(gdf_export).to_parquet(output_dir / f"{stem}.parquet", index=False)
    if "geojson" in export_formats:
        gdf_export.to_file(output_dir / f"{stem}.geojson", driver="GeoJSON")
    if "gpkg" in export_formats:
        gdf_export.to_file(output_dir / f"{stem}.gpkg", driver="GPKG")

## 1 · Charger Area Of Interest

3. Union des géométries multiples si nécessaire

**Zone d'étude (Area of Interest) :**2. Conversion en EPSG:4326 (coordonnées géographiques)

- **Mode shapefile** : Charge un périmètre depuis un fichier SIG existant1. Lecture/géocodage de la zone d'étude

- **Mode geocode** : Délimite automatiquement via OpenStreetMap (ex: "Genève, Switzerland")**Processus :**


In [5]:

aoi = load_aoi(area_mode, shapefile_path, places)
aoi_poly = aoi.geometry.iloc[0]
aoi


,name,geometry
0,aoi,"POLYGON ((6.15995 46.54286, 6.16101 46.54184, ..."


## 2 · Télécharger large (ways + nodes) via features_from_polygon

- Évite les biais de routabilité d'OSMnx

**Stratégie de téléchargement exhaustif :**- Permet classification personnalisée post-téléchargement

- **Ways** : Tous les segments `highway=*` (routes, chemins, pistes)- Inclut footways, paths, cycleways souvent exclus

- **Nodes** : Points de traversée `highway=crossing`- Récupère TOUS les éléments OSM, même non-routables

**Avantages vs `graph_from_polygon()` :**

In [6]:
print(type(aoi_poly))
print('Geometry type needs to be Polygon (or MultiPolygon)')
print(aoi_poly.geom_type)
print(aoi_poly.is_valid)

<class 'shapely.geometry.polygon.Polygon'>
Geometry type needs to be Polygon (or MultiPolygon)
Polygon
True


In [7]:
import time
import geopandas as gpd
import osmnx as ox
from osm_source import OSM_SOURCE, load_osm_features


TAGS_WAYS = {"highway": True}
TAGS_NODES = {"highway": "crossing"}

def fetch_osm_features_with_retry(
    polygon,
    tags,
    label: str,
    *,
    max_attempts: int = 4,
    timeout_s: int = 300,
    base_pause_s: float = 6.0,
    endpoints: list[str] | None = None,
    raise_on_failure: bool = True,
    geometry_types: set[str] | None = None,
) -> gpd.GeoDataFrame:

    if OSM_SOURCE == "local":
        print(f"[{label}] lecture locale")
        return load_osm_features(polygon, tags, territory, geometry_types,variant="network")


    
    """Télécharge des features OSM avec retries + fallback d'endpoint Overpass."""
    if endpoints is None:
        endpoints = [
            "https://overpass-api.de/api",
            "https://overpass.kumi.systems/api",
            "https://overpass.private.coffee/api",
        ]

    endpoint_attr = "overpass_url" if hasattr(ox.settings, "overpass_url") else "overpass_endpoint"
    previous_endpoint = getattr(ox.settings, endpoint_attr, None)
    previous_timeout = ox.settings.requests_timeout
    last_error = None

    try:
        for attempt in range(1, max_attempts + 1):
            endpoint = endpoints[(attempt - 1) % len(endpoints)]
            setattr(ox.settings, endpoint_attr, endpoint)
            ox.settings.requests_timeout = timeout_s

            try:
                gdf = ox.features_from_polygon(polygon, tags).reset_index()
                gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs="EPSG:4326")
                if geometry_types:
                    gdf = gdf[gdf.geometry.type.isin(sorted(geometry_types))].copy()
                return gdf
            except Exception as exc:
                last_error = exc
                wait_s = base_pause_s * (2 ** (attempt - 1))
                print(
                    f"[{label}] tentative {attempt}/{max_attempts} échouée sur {endpoint}: {type(exc).__name__}."
                )
                if attempt < max_attempts:
                    print(f"[{label}] nouvelle tentative dans {wait_s:.0f}s...")
                    time.sleep(wait_s)

        if raise_on_failure:
            raise RuntimeError(
                f"Échec du téléchargement OSM pour '{label}' après {max_attempts} tentatives."
            ) from last_error

        print(f"[{label}] échec après retries, retour d'une couche vide.")
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs="EPSG:4326")

    finally:
        if previous_endpoint is not None:
            setattr(ox.settings, endpoint_attr, previous_endpoint)
        ox.settings.requests_timeout = previous_timeout

ways_raw = fetch_osm_features_with_retry(
    aoi_poly,
    TAGS_WAYS,
    label="ways/highway",
    max_attempts=4,
    timeout_s=300,
    geometry_types={"LineString", "MultiLineString"},
)
ways_raw = explode_lines(ways_raw)

# ID stable OSM
ways_raw["osm_id"] = ways_raw["element"].astype(str) + "/" + ways_raw["id"].astype(str)

nodes_cross = fetch_osm_features_with_retry(
    aoi_poly,
    TAGS_NODES,
    label="nodes/crossing",
    max_attempts=3,
    timeout_s=240,
    geometry_types={"Point"},
    raise_on_failure=False,
)

ways_raw.shape, nodes_cross.shape

[ways/highway] lecture locale
[nodes/crossing] lecture locale


((211098, 703), (17168, 58))

## 3 · Classifier marche et vélo par règles

**Logique robuste** : Gère les tags OSM incomplets ou ambigus

**Classification piéton :**
- **Dédiés** : `footway`, `pedestrian`, `steps`, `platform`
- **Chemins** : `path` (sentiers partagés)
- **Exclusions** : `foot=no`, `access=no/private`

**Classification vélo étendue :**
- **Pistes séparées** : `cycleway`, `cycleway=track/separated`
- **Bandes cyclables** : `cycleway=lane/shared_lane` (marquage au sol)
- **Chemins** : `path`, `track`, `bridleway` avec accès vélo
- **Aires spéciales** : zones piétonnes (`highway=pedestrian`) et zones de rencontre (`highway=living_street`)
- **Routes partagées** : `primary` à `residential` (circulation motorisée)
- **Exclusions** : `bicycle=no`, puis `highway=service` (allègement du graphe), `access=no/private`, autoroutes et `trunk` sans accès vélo explicite
- **Exclusions avancées** : Autoroutes, routes rapides sauf permission explicite

In [8]:
# --------------------------------------------------
# Section 3 · Classification marche / vélo à partir des ways OSM
# Objectif:
# · garder un download large (toutes les ways highway pertinentes)
# · classer marche et vélo par règles
# · assurer un proxy piéton quand sidewalk est renseigné même sans trottoir géométrisé
# --------------------------------------------------

import re
import numpy as np
import pandas as pd

# --------------------------------------------------
# Helpers maxspeed et tags CH (optionnel mais recommandé)
# --------------------------------------------------

def parse_maxspeed_kmh(v) -> float | None:
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, (list, tuple, set)):
        v = list(v)[0] if len(v) else None
    if v is None:
        return None

    s = str(v).lower().strip()
    s = s.replace("km/h", "").replace("kph", "").strip()

    # valeurs non numériques fréquentes
    if s in {"none", "signals", "variable"}:
        return None

    # ex: "30;50" -> prendre la première valeur
    if ";" in s:
        s = s.split(";")[0].strip()

    m = re.search(r"(\d+(?:\.\d+)?)", s)
    if not m:
        return None
    try:
        return float(m.group(1))
    except Exception:
        return None


def has_explicit_sidewalk(row) -> bool:
    """
    True si OSM indique un trottoir via sidewalk=* ou footway=sidewalk,
    y compris formes left/right/both/yes/separate.
    """
    sw = row.get("sidewalk", None)
    if isinstance(sw, (list, tuple, set)):
        sw = list(sw)[0] if len(sw) else None
    sw = str(sw).lower().strip() if sw not in (None, np.nan) else ""

    # sidewalk=no doit être strictement excluant
    if sw == "no":
        return False

    if sw in {"yes", "both", "left", "right", "separate"}:
        return True

    # certains contributeurs utilisent footway=sidewalk sur l'axe routier
    fw = row.get("footway", None)
    if isinstance(fw, (list, tuple, set)):
        fw = list(fw)[0] if len(fw) else None
    fw = str(fw).lower().strip() if fw not in (None, np.nan) else ""
    if fw == "sidewalk":
        return True

    return False


def is_zone_20(row) -> bool:
    """
    Détecte des encodages CH de zone 20 (si présents).
    Fonction conservatrice: renvoie True si tag explicite zone20.
    """
    # zone:maxspeed=CH:zone20
    z = row.get("zone:maxspeed", None)
    if isinstance(z, (list, tuple, set)):
        z = list(z)[0] if len(z) else None
    z = str(z).strip() if z not in (None, np.nan) else ""
    if z == "CH:zone20":
        return True

    # maxspeed:type=CH:zone20
    mst = row.get("maxspeed:type", None)
    if isinstance(mst, (list, tuple, set)):
        mst = list(mst)[0] if len(mst) else None
    mst = str(mst).strip() if mst not in (None, np.nan) else ""
    if mst == "CH:zone20":
        return True

    return False


# --------------------------------------------------
# Classification marche
# --------------------------------------------------

WALK_DEDICATED_HW = {"footway", "pedestrian", "steps", "corridor", "platform"}
WALK_PATHLIKE_HW = {"path"}
WALK_TRACK_HW = {"track"}


# highways routiers qu'on accepte comme supports proxy quand sidewalk/zone apaisée
WALK_PROXY_ROAD_HW = {
    "motorway", "trunk", "primary", "secondary", "tertiary",
    "motorway_link", "trunk_link", "primary_link", "secondary_link", "tertiary_link",
    "unclassified", "residential", "living_street", "service",
}

def classify_walk(row) -> str:
    hw = str(row.get("highway", "")).lower()
    foot = str(row.get("foot", "")).lower()
    access = str(row.get("access", "")).lower()

    # exclusions strictes uniquement si marche explicitement interdite
    if foot == "no":
        return "exclude"

    # access=no/private excluant sauf si foot override
    if access in {"no", "private"} and foot not in {"yes", "designated", "permissive"}:
        return "exclude"

    # dédié explicite
    if hw in WALK_DEDICATED_HW:
        return "walk_dedicated"

    # path
    if hw in WALK_PATHLIKE_HW:
        return "walk_path"
        # path

    # track (selon TRACK_WALK_MODE)
    if hw in WALK_TRACK_HW and TRACK_WALK_MODE == "walk_track":
        return "walk_track"

    # proxy piéton si trottoir explicite sur un axe routier
    if hw in WALK_PROXY_ROAD_HW and has_explicit_sidewalk(row):
        return "walk_proxy_sidewalk"

    # proxy piéton si zone 20 explicite ou maxspeed <= 20
    # (si tu veux uniquement zone 20, commente le test maxspeed)
    if hw in WALK_PROXY_ROAD_HW:
        v = parse_maxspeed_kmh(row.get("maxspeed", None))
        if is_zone_20(row) or (v is not None and v <= 20):
            return "walk_proxy_zone20"

    return "other"


# --------------------------------------------------
# Classification vélo
# --------------------------------------------------

def classify_bike(row) -> str:
    hw = str(row.get("highway", "")).lower()
    bicycle = str(row.get("bicycle", "")).lower()
    access = str(row.get("access", "")).lower()

    # >>> DISMOUNT (Claude, 2026-11-25) : pied à terre obligatoire -> hors réseau vélo
    # AVANT : if bicycle == "no":  (dismount restait dans le graphe, surtout en bike_pedestrian_area à 0.4)
    if bicycle in {"no", "dismount"}:
        return "exclude"
    # <<< FIN DISMOUNT

    # Routes de service (accès, parkings, cours) : exclues pour alléger le graphe
    if hw == "service":
        return "exclude"

    if access in {"no", "private"} and bicycle not in {"yes", "designated", "permissive"}:
        return "exclude"

    if hw == "cycleway":
        return "bike_cycleway"

    cw = str(row.get("cycleway", "")).lower()
    cwl = str(row.get("cycleway:left", "")).lower()
    cwr = str(row.get("cycleway:right", "")).lower()
    cwb = str(row.get("cycleway:both", "")).lower()

    # pistes cyclables séparées
    if any("track" in x or "separated" in x for x in (cw, cwl, cwr, cwb)):
        return "bike_track"

    # bandes cyclables
    if any("lane" in x or "advisory" in x for x in (cw, cwl, cwr, cwb)):
        return "bike_lane"

    # shared
    if any("shared" in x for x in (cw, cwl, cwr, cwb)):
        return "bike_shared"

    if hw in {"path", "track", "bridleway"}:
        if bicycle in {"designated", "yes"}:
            return "bike_path_designated"
        return "bike_path"

    # Aires piétonnes et zones de rencontre (bicycle=no déjà exclu plus haut)
    if hw in {"pedestrian", "living_street"}:
        return "bike_pedestrian_area"

    # >>> FOOTWAY VÉLO (Claude, 2026-11-19) : trottoirs et chemins piétons ouverts aux vélos
    # AVANT : highway=footway tombait en "other" et sortait du graphe vélo (~200 km sur GG)
    if hw == "footway" and bicycle in {"designated", "yes", "permissive"}:
        return "bike_footway_designated" if bicycle == "designated" else "bike_footway_allowed"
    # <<< FIN FOOTWAY VÉLO

    motorized_highways = {
        "motorway", "trunk", "primary", "secondary", "tertiary",
        "motorway_link", "trunk_link", "primary_link", "secondary_link", "tertiary_link",
        "unclassified", "residential",
    }
    if hw in motorized_highways:
        if hw in {"motorway", "motorway_link"} and bicycle not in {"yes", "designated"}:
            return "exclude"
        if hw in {"trunk", "trunk_link"} and bicycle not in {"yes", "designated", "permissive"}:
            return "exclude"
        return "bike_road"

    return "other"


# --------------------------------------------------
# Application aux ways + extraction des sous-jeux
# --------------------------------------------------

ways = ways_raw.copy()

ways["walk_class"] = ways.apply(classify_walk, axis=1)
ways["bike_class"] = ways.apply(classify_bike, axis=1)

# Piéton:
# · dédié = walk_dedicated + walk_path
# · proxy = walk_proxy_sidewalk + walk_proxy_zone20
WALK_DEDICATED_CLASSES = {"walk_dedicated", "walk_path", "walk_track"}
walk_dedicated = ways[ways["walk_class"].isin(WALK_DEDICATED_CLASSES)].copy()
walk_proxy = ways[ways["walk_class"].isin({"walk_proxy_sidewalk", "walk_proxy_zone20"})].copy()

# Vélo:
bike_valid_classes = {
    "bike_cycleway", "bike_track", "bike_lane", "bike_shared",
    "bike_path", "bike_path_designated", "bike_pedestrian_area",
    "bike_footway_designated", "bike_footway_allowed",   # >>> FOOTWAY VÉLO (2026-11-19)
    "bike_road",
}
bike_base = ways[ways["bike_class"].isin(bike_valid_classes)].copy()

walk_dedicated.shape, walk_proxy.shape, bike_base.shape

((68496, 705), (9782, 705), (72215, 705))

## 4 · Consolider la marche

La couche marche combine trois familles d'objets :

- `walk_dedicated` et `walk_path` issus des ways OSM explicitement piétons ;
- `sidewalk_proxy_road_axis` issus d'axes routiers utilisés comme proxy de trottoir ou d'espace marchable ;
- objets optionnels : bords de plateformes et traversées synthétiques.

Le nettoyage des doublons entre proxies et infrastructures dédiées est appliqué plus loin, une fois les géométries converties dans un CRS métrique.


In [9]:
from shapely.geometry import LineString
import geopandas as gpd
import pandas as pd
import osmnx as ox


TRAM_BUS_KEYS = ("tram", "bus", "trolleybus")
TRAIN_KEYS = ("train", "light_rail", "subway")


def platform_axis(poly, min_len_m: float = 2.0):
    """Axe long d'un quai : ligne joignant les milieux des petits côtés du
    rectangle orienté minimal, découpée par le polygone (CRS métrique)."""
    if poly is None or poly.is_empty:
        return None
    rect = poly.minimum_rotated_rectangle
    if rect.geom_type != "Polygon":
        return None
    p = list(rect.exterior.coords)[:4]
    side01 = LineString([p[0], p[1]])
    side12 = LineString([p[1], p[2]])
    if side01.length >= side12.length:          # 0-1 = grand côté → axe entre 1-2 et 3-0
        a = side12.interpolate(0.5, normalized=True)
        b = LineString([p[3], p[0]]).interpolate(0.5, normalized=True)
    else:                                        # 1-2 = grand côté → axe entre 0-1 et 2-3
        a = side01.interpolate(0.5, normalized=True)
        b = LineString([p[2], p[3]]).interpolate(0.5, normalized=True)

    axis = LineString([a, b]).intersection(poly)
    if axis.is_empty:
        return None
    if axis.geom_type == "MultiLineString":      # quai courbe : on garde le plus long morceau
        axis = max(axis.geoms, key=lambda g: g.length)
    if axis.geom_type != "LineString" or axis.length < min_len_m:
        return None
    return axis


def _tag(row, key) -> str:
    v = row.get(key, None)
    if isinstance(v, (list, tuple, set)):
        v = next(iter(v), None)
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    return str(v).strip().lower()

def classify_platform(row) -> str:
    if any(_tag(row, k) == "yes" for k in TRAM_BUS_KEYS):
        return "tram_bus"
    if any(_tag(row, k) == "yes" for k in TRAIN_KEYS):
        return "train"
    if _tag(row, "railway") == "platform":
        return "train"
    return "tram_bus"   # highway=platform, bus_stop ou inconnu → vitesse adjacente

def platform_edges_from_polygon(aoi_poly) -> gpd.GeoDataFrame:
    tags = {"public_transport": "platform"}
    out_cols = ["geometry", "walk_class", "source", "highway", "platform_type", "osm_id"]
    empty = gpd.GeoDataFrame(columns=out_cols, geometry="geometry", crs="EPSG:4326")

    try:
        plats = load_osm_features(aoi_poly, tags, territory, variant="network")
    except Exception as exc:
        print(f"⚠️ quais non chargés : {type(exc).__name__}: {exc}")
        return empty

    plats = gpd.GeoDataFrame(plats, geometry="geometry", crs="EPSG:4326")
    if "element" in plats.columns and "id" in plats.columns:
        plats["osm_id"] = plats["element"].astype(str) + "/" + plats["id"].astype(str)
    else:
        plats["osm_id"] = None

    plats = plats[plats.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()
    plats = plats.explode(index_parts=False, ignore_index=True)
    plats["platform_type"] = plats.apply(classify_platform, axis=1)

    plats = to_metric_crs(plats, operation_crs)
    plats["geometry"] = plats.geometry.apply(platform_axis)
    plats = plats[plats.geometry.notna()].copy()

    plats["walk_class"] = "walk_dedicated"
    plats["highway"] = "platform"
    plats["source"] = "OSM_platform_polygon"

    print("Quais (axes) :", plats["platform_type"].value_counts().to_dict())
    return plats.to_crs("EPSG:4326")[out_cols].copy()


def has_sidewalk_tag(v) -> bool:
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return False
    if isinstance(v, (list, tuple, set)):
        v = list(v)[0] if len(v) else None
    if v is None:
        return False
    s = str(v).strip().lower()
    return s not in {"", "no", "none", "separate", "separated"}  # à adapter si besoin


def sidewalk_proxy_from_roads(ways: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    roads = ways[ways["highway"].isin(proxy_include_highways)].copy()
    if roads.empty:
        return gpd.GeoDataFrame(columns=["geometry","walk_class","source","osm_id"], geometry="geometry", crs=ways.crs)

    # signal sidewalk
    if "sidewalk" in roads.columns:
        roads["_has_sidewalk"] = roads["sidewalk"].apply(has_sidewalk_tag)
    else:
        roads["_has_sidewalk"] = False

    # vitesse parsée
    if "maxspeed" in roads.columns:
        roads["_maxspeed_kmh"] = roads["maxspeed"].apply(parse_maxspeed_kmh)
    else:
        roads["_maxspeed_kmh"] = None

    # règle d’inclusion
    # · si sidewalk explicite · inclure même si maxspeed est élevé
    # · sinon · garder la logique vitesse ≤ seuil (si parsable)
    if proxy_maxspeed_kmh is not None:
        m_speed = roads["_maxspeed_kmh"].isna() | (roads["_maxspeed_kmh"] <= float(proxy_maxspeed_kmh))
    else:
        m_speed = pd.Series(True, index=roads.index)

    m_keep = roads["_has_sidewalk"] | m_speed

    # si tu veux exiger un sidewalk explicite, garde seulement ce cas
    if proxy_require_sidewalk_tag:
        m_keep = roads["_has_sidewalk"]

    roads = roads[m_keep].copy()
    roads = roads.drop(columns=["_has_sidewalk","_maxspeed_kmh"], errors="ignore")

    roads["walk_class"] = "sidewalk_proxy_road_axis"
    roads["source"] = "OSM_highway_proxy"

    keep_cols = ["geometry","walk_class","source","highway","osm_id"]
    for c in ["id","element","sidewalk","maxspeed","name","service","access","foot"]:
        if c in roads.columns:
            keep_cols.append(c)

    return roads[keep_cols].copy()


def build_crossing_edges(
    nodes_cross: gpd.GeoDataFrame,
    roads: gpd.GeoDataFrame,
    operation_crs: str,
) -> gpd.GeoDataFrame:
    """Construit des traversées synthétiques. Les arêtes n'ont pas d'osm_id (None)."""
    out_cols = ["geometry", "walk_class", "source", "osm_id"]

    if nodes_cross.empty or roads.empty:
        return gpd.GeoDataFrame(columns=out_cols, geometry="geometry", crs="EPSG:4326")

    nodes = to_metric_crs(nodes_cross, operation_crs)
    r = to_metric_crs(roads, operation_crs)
    r = r[r.geometry.type.isin(["LineString", "MultiLineString"])].copy()
    if r.empty:
        return gpd.GeoDataFrame(columns=out_cols, geometry="geometry", crs="EPSG:4326")

    sindex = r.sindex
    segs = []

    for pt in nodes.geometry:
        if pt is None or pt.is_empty:
            continue

        buf = pt.buffer(float(crossing_snap_dist_m))
        cand_idx = list(sindex.intersection(buf.bounds))
        if not cand_idx:
            continue

        cand = r.iloc[cand_idx].copy()
        cand["dist"] = cand.distance(pt)
        cand = cand[cand["dist"] <= float(crossing_snap_dist_m)].sort_values("dist")
        if cand.empty:
            continue

        road = cand.iloc[0].geometry
        proj_d = road.project(pt)
        proj = road.interpolate(proj_d)

        # approx tangente locale
        delta = 1.0
        d1 = max(proj_d - delta, 0.0)
        d2 = min(proj_d + delta, road.length)
        p1 = road.interpolate(d1)
        p2 = road.interpolate(d2)

        vx = p2.x - p1.x
        vy = p2.y - p1.y
        norm = (vx**2 + vy**2) ** 0.5
        if norm == 0:
            continue
        vx /= norm
        vy /= norm

        # normale
        nx_ = -vy
        ny_ = vx

        hw = float(crossing_halfwidth_m)
        a = (proj.x - nx_ * hw, proj.y - ny_ * hw)
        b = (proj.x + nx_ * hw, proj.y + ny_ * hw)
        segs.append(LineString([a, b]))

    if not segs:
        return gpd.GeoDataFrame(columns=out_cols, geometry="geometry", crs="EPSG:4326")

    gdf = gpd.GeoDataFrame(
        {
            "walk_class": ["crossing_synthetic"] * len(segs),
            "source": ["OSM_crossing_node_proxy"] * len(segs),
            "osm_id": [None] * len(segs),
        },
        geometry=segs,
        crs=operation_crs,
    )

    return gdf.to_crs("EPSG:4326")[out_cols].copy()
    
walk_parts = []

# dédiés
wd = walk_dedicated.copy()
wd["source"] = "OSM_highway_dedicated"
wd["platform_type"] = None
m_plat = wd["highway"].eq("platform")
wd.loc[m_plat, "platform_type"] = wd[m_plat].apply(classify_platform, axis=1)


keep_cols = ["geometry","walk_class","source","highway","osm_id"]
for c in ["id","element","footway","sidewalk","name","maxspeed","service","access","foot", "platform_type", *TRACK_EXTRA_COLS]:
    if c in wd.columns:
        keep_cols.append(c)

walk_parts.append(wd[keep_cols])

if add_platform_edges:
    walk_parts.append(platform_edges_from_polygon(aoi_poly))

roads_for_crossings = ways[ways["highway"].isin(crossing_min_road_classes)].copy()

if add_sidewalk_proxy:
    walk_parts.append(sidewalk_proxy_from_roads(ways))

if add_crossing_edges:
    walk_parts.append(build_crossing_edges(nodes_cross, roads_for_crossings, operation_crs))

walk_edges = gpd.GeoDataFrame(pd.concat(walk_parts, ignore_index=True), geometry="geometry", crs="EPSG:4326")
walk_edges = walk_edges[walk_edges.geometry.notna()].copy()
walk_edges.shape

Quais (axes) : {'train': 158, 'tram_bus': 146}


(217184, 18)

## 5 · Consolider vélo (classification robuste)

**Typologie `infra_bike` (tags OSM, la première règle vraie l'emporte) :**
- **piste_cyclable** : `highway=cycleway`, ou `cycleway` / `cycleway:right` / `cycleway:left` / `cycleway:both` = `separate` ou `track`
- **bande_cyclable** : une de ces clés = `lane`
- **partage_bus** : une de ces clés = `share_busway` (ou `shared_busway`) ou `shared_lane`
- **bike_pedestrian_area** : `highway=pedestrian` ou `living_street`
- **partage** : `highway=track` (et `path`, `bridleway`)
- **bike_road** : autres routes

**`infra_cyclable`** (section 7, après le découpage) = la meilleure classe entre `infra_bike` et le schéma cyclable d'agglomération (`AGGLO_SCHEMA_CYCLABLE`, champ `amgt_exist`) lu dans un rectangle autour de chaque arête, comme la charge TIM.

**Gestion avancée des exclusions :**
- **Autoroutes** : Exclues sauf `bicycle=yes` explicite
- **Routes rapides** : `trunk/trunk_link` exclues par défaut
- **Hiérarchie des tags** : `bicycle=*` override `access=no`

**Gestion des sens de circulation :**
- **Analyse OSM** : Tags `oneway:bicycle` et `oneway`
- **3 directions** : `oneway`, `reverse`, `both`
- **Usage** : Préparation pour graphes orientés réalistes

In [10]:
# --------------------------------------------------
# Typologie infra_bike (tags OSM) puis infra_cyclable (OSM ∪ schéma d'agglomération)
# --------------------------------------------------
CYCLEWAY_KEYS = ["cycleway", "cycleway:right", "cycleway:left", "cycleway:both"]
# >>> SEPARATE (Claude, 2026-11-19) : cycleway=separate veut dire que la piste est dessinée comme une way à part.
# La route ne porte donc pas la piste : elle n'est plus classée piste_cyclable, et le filtre des doublons
# parallèles de 3_1 la retire là où la piste séparée la longe.
# AVANT : PISTE_CYCLEWAY_VALUES = {"separate", "track"}
PISTE_CYCLEWAY_VALUES = {"track"}            # track : piste sur le côté de la chaussée
# <<< FIN SEPARATE
BANDE_CYCLEWAY_VALUES = {"lane"}
# >>> BUS / CHAUSSÉE PARTAGÉE / CONTRESENS (Claude, 2026-11-23)
# AVANT : BUS_CYCLEWAY_VALUES = {"share_busway", "shared_busway", "shared_lane"}
BUS_CYCLEWAY_VALUES = {"share_busway", "shared_busway"}   # share_busway = valeur OSM documentée
# Voies bus décrites par busway:* (lane, opposite_lane...) ou par les voies (bus:lanes, psv:lanes...) -> partage_bus
BUSWAY_KEYS = ["busway", "busway:left", "busway:right", "busway:both"]
BUS_LANES_KEYS = ["bus:lanes", "bus:lanes:forward", "bus:lanes:backward",
                  "psv:lanes", "psv:lanes:forward", "psv:lanes:backward"]
SHARED_LANE_CYCLEWAY_VALUES = {"shared_lane"}            # marquage de chaussée partagée (pictogrammes), score 0.2
CONTRESENS_CYCLEWAY_VALUES = {"opposite_lane", "opposite_track"}   # bande / piste à contresens (tagging ancien)
# <<< FIN BUS / CHAUSSÉE PARTAGÉE / CONTRESENS
PARTAGE_HIGHWAYS = {"track", "path", "bridleway"}         # chemins agricoles et sentiers ouverts au vélo
# >>> NOUVELLES CLASSES (Claude, 2026-11-19)
SHOULDER_CYCLEWAY_VALUES = {"shoulder"}                   # accotement cyclable (routes départementales FR)

# Rang : en cas de recouvrement, la classe de rang le plus haut l'emporte (ordre = score dans 2_1)
# AVANT : {"piste_cyclable": 5, "bande_cyclable": 4, "partage_bus": 3, "bike_pedestrian_area": 2, "partage": 1, "bike_road": 0}
# AVANT (19.11) : {"piste_cyclable": 8, "bande_cyclable": 7, "voie_mixte": 6, "footway_designe": 5,
#                  "partage_bus": 4, "accotement": 3, "bike_pedestrian_area": 2, "partage": 1, "bike_road": 0}
INFRA_RANK = {"piste_cyclable": 10, "bande_cyclable": 9, "voie_mixte": 8, "footway_designe": 7,
              "contresens": 6, "partage_bus": 5, "accotement": 4, "bike_pedestrian_area": 3,
              "partage": 2, "chaussee_partagee": 1, "bike_road": 0}   # >>> + contresens, chaussee_partagee (2026-11-23)
# <<< FIN NOUVELLES CLASSES


def _tag_values(row, keys):
    """Valeurs des tags (séparées par ';'), en minuscules."""
    vals = set()
    for k in keys:
        v = row.get(k, None)
        if v is None or (isinstance(v, float) and np.isnan(v)):
            continue
        vals |= {x.strip().lower() for x in str(v).split(";")}
    return vals


def map_bike_infra(row) -> str:
    """Typologie infra_bike à partir des seuls tags OSM (ordre = priorité)."""
    hw = str(row.get("highway", "")).lower()
    cw = _tag_values(row, CYCLEWAY_KEYS)
    # >>> NOUVELLES CLASSES (Claude, 2026-11-19)
    # voie_mixte      : voie dédiée vélo partagée avec les piétons (path + bicycle=designated non séparé,
    #                   cycleway + segregated=no)
    # footway_designe : chemin / trottoir piéton avec bicycle=designated
    # accotement      : cycleway=shoulder
    # footway + bicycle=yes/permissive -> bike_pedestrian_area
    bicycle = str(row.get("bicycle", "")).lower()
    segregated = str(row.get("segregated", "")).lower()
    bike_class = row.get("bike_class")

    # AVANT : if hw == "cycleway" or cw & PISTE_CYCLEWAY_VALUES: return "piste_cyclable"
    if hw == "cycleway":
        return "voie_mixte" if segregated == "no" else "piste_cyclable"
    # AVANT : tout highway=path tombait en "partage", même avec bicycle=designated
    if hw == "path" and bicycle == "designated":
        return "piste_cyclable" if segregated == "yes" else "voie_mixte"
    if cw & PISTE_CYCLEWAY_VALUES:
        return "piste_cyclable"
    # <<< FIN NOUVELLES CLASSES
    if cw & BANDE_CYCLEWAY_VALUES:
        return "bande_cyclable"
    # >>> BUS / CHAUSSÉE PARTAGÉE / CONTRESENS (Claude, 2026-11-23)
    if cw & CONTRESENS_CYCLEWAY_VALUES:
        return "contresens"
    # AVANT : if cw & BUS_CYCLEWAY_VALUES: return "partage_bus"
    busway = _tag_values(row, BUSWAY_KEYS) - {"no", "none", "nan", ""}
    bus_lanes = any("designated" in str(row.get(k, "")).lower() for k in BUS_LANES_KEYS)
    if cw & BUS_CYCLEWAY_VALUES or busway or bus_lanes:
        return "partage_bus"
    # <<< FIN BUS / CHAUSSÉE PARTAGÉE / CONTRESENS
    # >>> NOUVELLES CLASSES (Claude, 2026-11-19)
    if cw & SHOULDER_CYCLEWAY_VALUES:
        return "accotement"
    if bike_class == "bike_footway_designated":
        return "footway_designe"
    # AVANT : if row.get("bike_class") == "bike_pedestrian_area":
    if bike_class in {"bike_pedestrian_area", "bike_footway_allowed"}:
        return "bike_pedestrian_area"
    # <<< FIN NOUVELLES CLASSES
    if hw in PARTAGE_HIGHWAYS:
        return "partage"
    # >>> CHAUSSÉE PARTAGÉE (Claude, 2026-11-23) : cycleway=shared_lane (marquage), après "partage" pour ne pas
    # déclasser un chemin ; AVANT : shared_lane était dans BUS_CYCLEWAY_VALUES -> partage_bus
    if cw & SHARED_LANE_CYCLEWAY_VALUES:
        return "chaussee_partagee"
    # <<< FIN CHAUSSÉE PARTAGÉE
    return "bike_road"

def get_bike_direction(row) -> str:
    """
    Détermine le sens de circulation vélo selon la logique OSM.
    Retourne: 'oneway', 'reverse', 'both'
    """
    oneway_bike = str(row.get("oneway:bicycle", "")).lower()
    oneway = str(row.get("oneway", "")).lower()
    
    # Priorité à oneway:bicycle si défini
    if oneway_bike == "yes":
        return "oneway"
    elif oneway_bike == "-1":
        return "reverse"
    elif oneway_bike == "no":
        return "both"
    
    # Sinon, hériter du oneway général
    if oneway == "yes":
        return "oneway"
    elif oneway == "-1":
        return "reverse"
    
    # Par défaut: double sens
    return "both"

bike_edges = bike_base.copy()
bike_edges["infra_bike"] = bike_edges.apply(map_bike_infra, axis=1)
bike_edges["bike_direction"] = bike_edges.apply(get_bike_direction, axis=1)
bike_edges["source"] = "OSM_highway_bike"

keep_cols = ["geometry","infra_bike","bike_direction","source","highway"]
for c in ["cycleway","cycleway:left","cycleway:right","cycleway:both","bicycle","segregated","busway","busway:left","busway:right","busway:both","bus:lanes","psv:lanes","oneway","oneway:bicycle","name","maxspeed","maxspeed:forward","maxspeed:backward","zone:maxspeed","maxspeed:type","surface","osmid"]:  # >>> + segregated (2026-11-19)
    if c in bike_edges.columns:
        keep_cols.append(c)

bike_edges = gpd.GeoDataFrame(bike_edges[keep_cols].copy(), geometry="geometry", crs="EPSG:4326")
bike_edges.shape

(72215, 25)

## 6 · Nettoyage des géométries marche/vélo

Cette section prépare les géométries avant la segmentation du graphe.

Le filtre principal de cette version bis supprime uniquement les `sidewalk_proxy_road_axis` qui font doublon avec des `walk_dedicated` proches. La règle est volontairement simple : un proxy est supprimé si une part suffisante de sa longueur est couverte par le buffer des lignes dédiées proches.

Les proxies sans infrastructure dédiée proche sont conservés, ce qui préserve leur utilité dans les zones périphériques ou peu cartographiées.


In [11]:
# --------------------------------------------------
# Section 6 · Consolidation finale réseau piéton/vélo
# --------------------------------------------------
from shapely.ops import unary_union

# Convertir en CRS métrique pour les opérations SIG
walk_edges_p = to_metric_crs(walk_edges, operation_crs)
bike_edges_p = to_metric_crs(bike_edges, operation_crs)

walk_edges_p = explode_lines(walk_edges_p)
bike_edges_p = explode_lines(bike_edges_p)

walk_edges_p = simplify_gdf(walk_edges_p, simplify_tol_m, operation_crs)
bike_edges_p = simplify_gdf(bike_edges_p, simplify_tol_m, operation_crs)

# Longueurs métriques
walk_edges_p["len_m"] = walk_edges_p.length
bike_edges_p["len_m"] = bike_edges_p.length


# --------------------------------------------------
# walk_role · dedicated | crossing | proxy_primary | proxy_local
# --------------------------------------------------
motor_axes = {
    "primary", "secondary", "tertiary",
    "primary_link", "secondary_link", "tertiary_link",
}


def assign_walk_role(row) -> str:
    wc = str(row.get("walk_class", ""))
    hw = str(row.get("highway", ""))

    if wc == "crossing_synthetic":
        return "crossing"

    if wc == proxy_duplicate_class:
        if hw in motor_axes:
            return "proxy_primary"
        return "proxy_local"

    return "dedicated"


walk_edges_p["walk_role"] = walk_edges_p.apply(assign_walk_role, axis=1)


# --------------------------------------------------
# Filtre simple des proxies redondants
# --------------------------------------------------
def filter_redundant_sidewalk_proxies(
    walk_edges_metric: gpd.GeoDataFrame,
    proxy_class: str,
    dedicated_classes: set[str],
    buffer_m: float,
    min_covered_len_m: float,
    min_coverage_ratio: float,
    min_dedicated_len_m: float = 0.0,
) -> tuple[gpd.GeoDataFrame, gpd.GeoDataFrame, dict]:
    """
    Supprime les proxies dont la géométrie est déjà couverte par des lignes dédiées proches.

    Critère de suppression : longueur du proxy incluse dans le buffer des dedicated
    supérieure à un seuil absolu et à un ratio minimal de la longueur du proxy.
    """
    if walk_edges_metric.empty:
        summary = {"proxy_initial": 0, "proxy_removed": 0, "proxy_kept": 0}
        removed = gpd.GeoDataFrame(columns=walk_edges_metric.columns, geometry="geometry", crs=walk_edges_metric.crs)
        return walk_edges_metric.copy(), removed, summary

    edges = walk_edges_metric.copy()
    proxy_mask = edges["walk_class"].eq(proxy_class)
    dedicated_mask = edges["walk_class"].isin(dedicated_classes)

    proxies = edges[proxy_mask].copy()
    dedicated = edges[dedicated_mask].copy()
    others = edges[~proxy_mask].copy()

    if proxies.empty or dedicated.empty:
        summary = {
            "proxy_initial": int(len(proxies)),
            "proxy_removed": 0,
            "proxy_kept": int(len(proxies)),
            "buffer_m": float(buffer_m),
            "min_covered_len_m": float(min_covered_len_m),
            "min_coverage_ratio": float(min_coverage_ratio),
        }
        removed = gpd.GeoDataFrame(columns=edges.columns, geometry="geometry", crs=edges.crs)
        return edges, removed, summary

    if min_dedicated_len_m and "len_m" in dedicated.columns:
        dedicated = dedicated[dedicated["len_m"] >= float(min_dedicated_len_m)].copy()
    if dedicated.empty:
        summary = {"proxy_initial": int(len(proxies)), "proxy_removed": 0, "proxy_kept": int(len(proxies))}
        removed = gpd.GeoDataFrame(columns=edges.columns, geometry="geometry", crs=edges.crs)
        return edges, removed, summary

    sidx = dedicated.sindex
    keep_flags = []
    covered_lengths = []
    coverage_ratios = []

    for _, row in proxies.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty or geom.length == 0:
            keep_flags.append(True)
            covered_lengths.append(0.0)
            coverage_ratios.append(0.0)
            continue

        search_geom = geom.buffer(float(buffer_m))
        candidate_idx = list(sidx.intersection(search_geom.bounds))
        if not candidate_idx:
            keep_flags.append(True)
            covered_lengths.append(0.0)
            coverage_ratios.append(0.0)
            continue

        near = dedicated.iloc[candidate_idx].copy()
        near = near[near.distance(geom) <= float(buffer_m)].copy()
        if near.empty:
            keep_flags.append(True)
            covered_lengths.append(0.0)
            coverage_ratios.append(0.0)
            continue

        cover_zone = unary_union(list(near.geometry)).buffer(float(buffer_m))
        intersection = geom.intersection(cover_zone)
        covered_len = intersection.length if not intersection.is_empty else 0.0
        coverage_ratio = covered_len / geom.length if geom.length > 0 else 0.0

        drop = (covered_len >= float(min_covered_len_m)) and (coverage_ratio >= float(min_coverage_ratio))
        keep_flags.append(not drop)
        covered_lengths.append(float(covered_len))
        coverage_ratios.append(float(coverage_ratio))

    proxies["proxy_dedicated_covered_len_m"] = covered_lengths
    proxies["proxy_dedicated_coverage_ratio"] = coverage_ratios

    kept_proxies = proxies.loc[keep_flags].copy()
    removed_proxies = proxies.loc[[not flag for flag in keep_flags]].copy()

    cleaned = pd.concat([others, kept_proxies], ignore_index=True)
    cleaned = gpd.GeoDataFrame(cleaned, geometry="geometry", crs=edges.crs)

    summary = {
        "proxy_initial": int(len(proxies)),
        "proxy_removed": int(len(removed_proxies)),
        "proxy_kept": int(len(kept_proxies)),
        "buffer_m": float(buffer_m),
        "min_covered_len_m": float(min_covered_len_m),
        "min_coverage_ratio": float(min_coverage_ratio),
    }
    return cleaned, removed_proxies, summary


removed_sidewalk_proxy_duplicates = gpd.GeoDataFrame(
    columns=walk_edges_p.columns,
    geometry="geometry",
    crs=walk_edges_p.crs,
)
proxy_duplicate_summary = {
    "proxy_initial": int((walk_edges_p["walk_class"] == proxy_duplicate_class).sum()),
    "proxy_removed": 0,
    "proxy_kept": int((walk_edges_p["walk_class"] == proxy_duplicate_class).sum()),
}

if remove_proxy_near_dedicated:
    walk_edges_p, removed_sidewalk_proxy_duplicates, proxy_duplicate_summary = filter_redundant_sidewalk_proxies(
        walk_edges_metric=walk_edges_p,
        proxy_class=proxy_duplicate_class,
        dedicated_classes=proxy_dedicated_classes,
        buffer_m=proxy_dedicated_buffer_m,
        min_covered_len_m=proxy_min_covered_len_m,
        min_coverage_ratio=proxy_min_coverage_ratio,
        min_dedicated_len_m=proxy_min_dedicated_len_m,
    )

print("Nettoyage proxies redondants:", proxy_duplicate_summary)


# --------------------------------------------------
# Déduplication géométrique finale
# --------------------------------------------------
if deduplicate_by_geometry:
    walk_edges_p["__wkb"] = walk_edges_p.geometry.to_wkb()
    walk_edges_p = walk_edges_p.drop_duplicates(subset="__wkb").drop(columns="__wkb")

    bike_edges_p["__wkb"] = bike_edges_p.geometry.to_wkb()
    bike_edges_p = bike_edges_p.drop_duplicates(subset="__wkb").drop(columns="__wkb")

walk_edges_p.shape, bike_edges_p.shape, removed_sidewalk_proxy_duplicates.shape

Nettoyage proxies redondants: {'proxy_initial': 148384, 'proxy_removed': 8977, 'proxy_kept': 139407, 'buffer_m': 12.0, 'min_covered_len_m': 20.0, 'min_coverage_ratio': 0.65}


((184278, 22), (72215, 26), (8977, 22))

## 7 · Topologie et graphe

### Graphe orienté vs non-orienté pour le vélo

**Graphe non-orienté (traditionnel) :**
- Chaque segment de route = 1 arête bidirectionnelle
- Navigation possible dans les 2 sens sur toutes les arêtes
- Simple mais **ignore les contraintes de circulation réelles**

**Graphe orienté (réaliste) :**
- Respecte les sens de circulation selon les tags OSM :
  - `oneway:bicycle=yes` → sens unique vélo (1 arête u→v)
  - `oneway:bicycle=no` → double sens forcé (2 arêtes u→v + v→u)  
  - `oneway=yes` → sens unique général hérité par le vélo
  - Par défaut → double sens (2 arêtes)

**Implications pratiques :**

1. **Routage réaliste** : Les itinéraires respectent les sens uniques cyclables
2. **Analyse de connectivité** : Identifie les culs-de-sac directionnels
3. **Mesures de centralité** : Calculs précis des flux directionnels
4. **Accessibilité** : Evaluation correcte des zones atteignables

**Cas typiques à Genève :**
- Rues en sens unique avec contre-sens cyclable
- Pistes cyclables unidirectionnelles  
- Zones 30 bidirectionnelles
- Axes motorisés avec aménagements asymétriques

**Usage :**
- `bike_directed_graph = True` : Analyses de mobilité réalistes
- `bike_directed_graph = False` : Analyses topologiques générales

In [12]:
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
from shapely.geometry import Point, LineString, MultiPoint
from shapely.ops import split
import networkx as nx

def _iter_intersection_points(geom):
    """
    Retourne une liste de Points à partir d'une géométrie d'intersection.
    Ignore les intersections de type LineString (superpositions).
    """
    if geom is None or geom.is_empty:
        return []
    gt = geom.geom_type
    if gt == "Point":
        return [geom]
    if gt == "MultiPoint":
        return list(geom.geoms)
    if gt == "GeometryCollection":
        pts = []
        for g in geom.geoms:
            pts.extend(_iter_intersection_points(g))
        return pts
    return []  # LineString, MultiLineString, Polygon, etc.

def split_edges_at_intersections(
    edges: gpd.GeoDataFrame,
    crs_metric: str,
    min_seg_len_m: float = 1.0,
) -> gpd.GeoDataFrame:
    """
    Split toutes les LineString aux intersections géométriques (y compris T-intersections).
    Retourne un nouveau GeoDataFrame d'arêtes segmentées (attributs dupliqués).
    """
    e = edges.to_crs(crs_metric).copy()
    e = e[e.geometry.notna()].copy()
    e = e[e.geometry.geom_type == "LineString"].copy()
    e = e.reset_index(drop=True)
    e["edge_id"] = np.arange(len(e), dtype=int)

    if e.empty:
        return e

    sidx = e.sindex
    split_pts_by_edge = {i: [] for i in e.index}

    # Détection intersections avec sindex, en évitant les doublons i<j
    for i, geom_i in zip(e.index, e.geometry):
        cand = list(sidx.intersection(geom_i.bounds))
        for j in cand:
            if j <= i:
                continue
            geom_j = e.geometry.iloc[j]
            if not geom_i.intersects(geom_j):
                continue
            inter = geom_i.intersection(geom_j)
            pts = _iter_intersection_points(inter)
            if not pts:
                continue

            # Ajouter à i et j
            split_pts_by_edge[i].extend(pts)
            split_pts_by_edge[j].extend(pts)

    # Split chaque ligne sur ses points
    out_rows = []
    for idx, row in e.iterrows():
        geom = row.geometry
        pts = split_pts_by_edge.get(idx, [])

        # ajouter endpoints pour stabiliser split (optionnel, mais utile)
        coords = list(geom.coords)
        pts.append(Point(coords[0]))
        pts.append(Point(coords[-1]))

        # dédoublonner les points (par WKB)
        if pts:
            wkb_seen = set()
            pts_u = []
            for p in pts:
                w = p.wkb
                if w not in wkb_seen:
                    wkb_seen.add(w)
                    pts_u.append(p)
            pts = pts_u

        if len(pts) <= 2:
            geoms = [geom]
        else:
            try:
                geoms = list(split(geom, MultiPoint(pts)).geoms)
            except Exception:
                geoms = [geom]

        for g in geoms:
            if g is None or g.is_empty:
                continue
            if g.length < float(min_seg_len_m):
                continue
            new_row = row.drop(labels=["geometry"]).to_dict()
            out_rows.append({**new_row, "geometry": g})

    out = gpd.GeoDataFrame(out_rows, geometry="geometry", crs=crs_metric)
    out = out.reset_index(drop=True)
    return out

def build_osmnx_graph_from_segmented_edges(
    edges_seg: gpd.GeoDataFrame,
    crs_metric: str,
    snap_precision_m: float = 0.5,
    directed: bool = False,
):
    e = edges_seg.to_crs(crs_metric).copy()
    e = e[e.geometry.notna()].copy()
    e = e[e.geometry.geom_type == "LineString"].copy()
    e = e.reset_index(drop=True)

    def _round_xy(x, y):
        return (
            round(x / snap_precision_m) * snap_precision_m,
            round(y / snap_precision_m) * snap_precision_m,
        )

    u_xy, v_xy = [], []
    for geom in e.geometry:
        coords = list(geom.coords)
        x1, y1 = coords[0]
        x2, y2 = coords[-1]
        u_xy.append(_round_xy(x1, y1))
        v_xy.append(_round_xy(x2, y2))

    uniq_xy = pd.Index(pd.unique(pd.Index(u_xy + v_xy)))
    nodes = gpd.GeoDataFrame(
        {"osmid": np.arange(len(uniq_xy), dtype=int)},
        geometry=[Point(x, y) for (x, y) in uniq_xy],
        crs=crs_metric,
    )
    nodes["x"] = nodes.geometry.x
    nodes["y"] = nodes.geometry.y
    nodes = nodes.set_index("osmid", drop=True)

    xy_to_id = {xy: int(i) for i, xy in enumerate(uniq_xy)}

    e["u"] = [xy_to_id[xy] for xy in u_xy]
    e["v"] = [xy_to_id[xy] for xy in v_xy]
    e["length"] = e.geometry.length

    # supprimer self-loops dégénérés
    e = e[e["u"] != e["v"]].copy()

    # key unique par (u,v)
    e["_key"] = e.groupby(["u", "v"]).cumcount()

    edges_gdf = e.set_index(["u", "v", "_key"], drop=True)
    edges_gdf.index = edges_gdf.index.set_names(["u", "v", "key"])

    if not nodes.index.is_unique:
        raise ValueError("nodes index not unique after snapping")
    if not edges_gdf.index.is_unique:
        raise ValueError("edges index (u,v,key) not unique, check snapping or duplicates")

    G = ox.graph_from_gdfs(nodes, edges_gdf)

    if not directed:
        G = nx.MultiGraph(G)

    edges_out = edges_gdf.reset_index()

    return G, nodes, edges_out

def build_directed_bike_graph_from_segmented_edges(
    edges_seg: gpd.GeoDataFrame,
    crs_metric: str,
    snap_precision_m: float = 0.5,
):
    """
    Construit un graphe orienté vélo en respectant les sens de circulation.
    """
    e = edges_seg.to_crs(crs_metric).copy()
    e = e[e.geometry.notna()].copy()
    e = e[e.geometry.geom_type == "LineString"].copy()
    e = e.reset_index(drop=True)

    def _round_xy(x, y):
        return (
            round(x / snap_precision_m) * snap_precision_m,
            round(y / snap_precision_m) * snap_precision_m,
        )

    # Expansion des arêtes selon leur direction
    expanded_edges = []
    
    for idx, row in e.iterrows():
        geom = row.geometry
        coords = list(geom.coords)
        x1, y1 = coords[0]
        x2, y2 = coords[-1]
        u_xy = _round_xy(x1, y1)
        v_xy = _round_xy(x2, y2)
        
        direction = row.get('bike_direction', 'both')
        
        # Créer les arêtes selon la direction
        if direction == 'oneway':
            # Sens normal uniquement (u -> v)
            expanded_edges.append({**row.drop('geometry'), 'u_xy': u_xy, 'v_xy': v_xy, 'geometry': geom})
        elif direction == 'reverse':
            # Sens inverse uniquement (v -> u)
            reversed_geom = LineString(coords[::-1])
            expanded_edges.append({**row.drop('geometry'), 'u_xy': v_xy, 'v_xy': u_xy, 'geometry': reversed_geom})
        else:  # 'both'
            # Double sens
            expanded_edges.append({**row.drop('geometry'), 'u_xy': u_xy, 'v_xy': v_xy, 'geometry': geom})
            reversed_geom = LineString(coords[::-1])
            expanded_edges.append({**row.drop('geometry'), 'u_xy': v_xy, 'v_xy': u_xy, 'geometry': reversed_geom})
    
    e_expanded = gpd.GeoDataFrame(expanded_edges, crs=crs_metric)
    
    # Créer les nœuds uniques
    all_xy = list(e_expanded['u_xy']) + list(e_expanded['v_xy'])
    uniq_xy = pd.Index(pd.unique(pd.Index(all_xy)))
    
    nodes = gpd.GeoDataFrame(
        {"osmid": np.arange(len(uniq_xy), dtype=int)},
        geometry=[Point(x, y) for (x, y) in uniq_xy],
        crs=crs_metric,
    )
    nodes["x"] = nodes.geometry.x
    nodes["y"] = nodes.geometry.y
    nodes = nodes.set_index("osmid", drop=True)

    xy_to_id = {xy: int(i) for i, xy in enumerate(uniq_xy)}

    # Assigner u, v aux arêtes expandues
    e_expanded["u"] = [xy_to_id[xy] for xy in e_expanded['u_xy']]
    e_expanded["v"] = [xy_to_id[xy] for xy in e_expanded['v_xy']]
    e_expanded["length"] = e_expanded.geometry.length
    
    # Nettoyer les colonnes temporaires
    e_expanded = e_expanded.drop(columns=['u_xy', 'v_xy'])

    # Supprimer self-loops
    e_expanded = e_expanded[e_expanded["u"] != e_expanded["v"]].copy()

    # Key unique par (u,v)
    e_expanded["_key"] = e_expanded.groupby(["u", "v"]).cumcount()

    edges_gdf = e_expanded.set_index(["u", "v", "_key"], drop=True)
    edges_gdf.index = edges_gdf.index.set_names(["u", "v", "key"])

    # Construire le graphe dirigé
    G = ox.graph_from_gdfs(nodes, edges_gdf)
    # G reste un MultiDiGraph pour le graphe orienté

    edges_out = edges_gdf.reset_index()

    return G, nodes, edges_out

In [13]:
# 1) segmenter aux intersections
walk_edges_seg = split_edges_at_intersections(
    walk_edges_p,
    crs_metric=operation_crs,
    min_seg_len_m=1.0,
)
print("Segmentation piéton terminée.")

bike_edges_seg = split_edges_at_intersections(
    bike_edges_p,
    crs_metric=operation_crs,
    min_seg_len_m=1.0,
)
print("Segmentation vélo terminée.")


Segmentation piéton terminée.
Segmentation vélo terminée.


In [14]:
# Vitesse de la chaussée adjacente, tirée de ways_raw (avant toute suppression)
ROAD_HW = {"motorway","trunk","primary","secondary","tertiary","unclassified",
           "residential","living_street","service",
           "motorway_link","trunk_link","primary_link","secondary_link","tertiary_link"}
SPEED_COLS = [c for c in ["maxspeed", "zone:maxspeed", "maxspeed:type"] if c in ways_raw.columns]

def speed_kmh(row, prefix=""):
    # maxspeed explicite d'abord, puis l'encodage de zone (ex. "CH:zone30" -> 30)
    for c in SPEED_COLS:
        v = parse_maxspeed_kmh(row.get(prefix + c))
        if v is not None:
            return v
    return None

# Relance de cellule sans erreur : retirer les colonnes déjà ajoutées
walk_edges_seg = walk_edges_seg.drop(
    columns=["adj_osm_id", "adj_highway", "adj_speed_kmh", "adj_dist_m", "speed_ctx_kmh"],
    errors="ignore",
)

roads = ways_raw[ways_raw["highway"].isin(ROAD_HW)][["osm_id", "highway", "geometry", *SPEED_COLS]]
roads = to_metric_crs(roads, operation_crs)
roads["adj_speed_kmh"] = roads.apply(speed_kmh, axis=1)
roads = roads.rename(columns={"osm_id": "adj_osm_id", "highway": "adj_highway"})[
    ["adj_osm_id", "adj_highway", "adj_speed_kmh", "geometry"]]

# Milieu de chaque segment piéton -> chaussée la plus proche (<= 15 m)
mids = walk_edges_seg[["geometry"]].copy()
mids["geometry"] = mids.geometry.interpolate(0.5, normalized=True)
j = gpd.sjoin_nearest(mids, roads, how="left", max_distance=15, distance_col="adj_dist_m")
j = j[~j.index.duplicated(keep="first")].drop(columns=["index_right", "geometry"])
walk_edges_seg = walk_edges_seg.join(j)

# Vitesse de contexte : le tag propre pour un proxy, sinon la chaussée adjacente
own = walk_edges_seg.apply(speed_kmh, axis=1)
is_proxy = walk_edges_seg["walk_class"].eq("sidewalk_proxy_road_axis")
walk_edges_seg["speed_ctx_kmh"] = np.where(is_proxy, own, walk_edges_seg["adj_speed_kmh"])
walk_edges_seg["speed_ctx_kmh"] = pd.to_numeric(walk_edges_seg["speed_ctx_kmh"], errors="coerce")

print(f"Vitesse de contexte renseignée : {walk_edges_seg['speed_ctx_kmh'].notna().mean():.1%} des segments piétons")


Vitesse de contexte renseignée : 22.4% des segments piétons


### Infrastructure cyclable : OSM ∪ schéma cyclable d'agglomération

`infra_cyclable` = la classe la plus favorable entre `infra_bike` (tags OSM) et le schéma cyclable d'agglomération (`AGGLO_SCHEMA_CYCLABLE.gpkg`, champ `amgt_exist`, aménagements existants uniquement).

Lecture du schéma : même méthode que la charge TIM (`feature_rectangle.py`). Rectangle à extrémités plates autour de chaque arête vélo découpée, demi-largeur `SCHEMA_HALF_WIDTH_M`, lignes à plus de 45° écartées ; l'arête prend la valeur du schéma qui couvre le plus de longueur, si elle couvre au moins `SCHEMA_MIN_SHARE` de l'arête.

| `amgt_exist` | classe |
| --- | --- |
| Site propre (voie verte ou piste cyclable) | piste_cyclable |
| Aménagement cyclable sur chaussée (marquage ou bande) | bande_cyclable |
| Route partagée sans nécessité d'aménagement | partage |
| Modération des vitesses | non utilisé |

**Garde sur les routes** *(ajout Claude, 2026-11-19)* : la ligne « Site propre » du schéma suit le tracé de la piste, souvent à 5–8 m de l'axe OSM de la route. Sans garde, la route tombe dans son rectangle et passe en `piste_cyclable`. Le schéma ne peut donc plus mettre en piste une **route** (highway hors `cycleway`, `path`, `footway`, `pedestrian`, `track`, `bridleway`) déjà longée par une voie cyclable OSM **dessinée à part** (`piste_cyclable`, `voie_mixte`, `footway_designe`) : rectangle `SCHEMA_ROAD_GUARD_HALF_WIDTH_M`, angle ≤ `SCHEMA_ROAD_GUARD_MAX_ANGLE_DEG`, part couverte ≥ `SCHEMA_ROAD_GUARD_MIN_SHARE`. Ces routes gardent leur classe OSM (colonne `schema_piste_ignoree` = True) et le filtre des doublons de `3_1` les retire. Une route sans piste OSM à côté garde la piste du schéma : c'est un aménagement absent d'OSM.

**Classes OSM protégées** *(ajout Claude, 2026-11-23)* : « Site propre » reste `piste_cyclable`, mais le schéma ne surclasse plus une arête qu'OSM classe `voie_mixte` ou `footway_designe` (`SCHEMA_PROTECTED_OSM`) : la catégorie du schéma mélange voies vertes et pistes, OSM est plus précis.


In [15]:
# ============================================================================
# infra_cyclable = max(infra_bike OSM, schéma cyclable d'agglomération)
# ============================================================================
import sys
sys.path.insert(0, str(Path("../Step 2").resolve()))     # feature_rectangle.py (Notebook/Step 2)
import importlib, feature_rectangle
importlib.reload(feature_rectangle)
from feature_rectangle import extract_rectangle_majority_class

SCHEMA_PATH = INPUT_ROOT / "network_agreg/AGGLO_SCHEMA_CYCLABLE.gpkg"
SCHEMA_CLASS_COL = "amgt_exist"            # amgt_existant, tronqué dans le gpkg
SCHEMA_TO_INFRA = {
    "Site propre (voie verte ou piste cyclable)": "piste_cyclable",
    "Aménagement cyclable sur chaussée (marquage ou bande)": "bande_cyclable",
    "Route partagée sans nécessité d'aménagement": "partage",
}
SCHEMA_HALF_WIDTH_M = 10.0      # demi-largeur du rectangle autour de l'arête
SCHEMA_MIN_SHARE = 0.5          # part minimale de l'arête couverte par la classe du schéma
SCHEMA_MAX_ANGLE_DEG = 45.0

if SCHEMA_PATH.exists():
    schema = gpd.read_file(SCHEMA_PATH).to_crs(operation_crs)
    schema["geometry"] = schema.geometry.force_2d()
    schema["infra_schema"] = schema[SCHEMA_CLASS_COL].map(SCHEMA_TO_INFRA)
    schema = schema[schema["infra_schema"].notna()].explode(index_parts=False, ignore_index=True)
    print(f"Schéma cyclable : {len(schema):,} lignes retenues")
    print((schema.length.groupby(schema["infra_schema"]).sum() / 1000).round(1).rename("km").to_string())

    bike_edges_seg["_eid"] = np.arange(len(bike_edges_seg))
    match = extract_rectangle_majority_class(
        bike_edges_seg[["_eid", "geometry"]], schema[["infra_schema", "geometry"]], "infra_schema",
        SCHEMA_HALF_WIDTH_M, max_angle_deg=SCHEMA_MAX_ANGLE_DEG, min_share=SCHEMA_MIN_SHARE,
        crs_meter_epsg=operation_crs, segment_id_col="_eid",
    )
    bike_edges_seg = bike_edges_seg.drop(columns="_eid")
    bike_edges_seg["infra_schema"] = match["infra_schema"].to_numpy()
    bike_edges_seg["infra_schema_share"] = match["infra_schema_share"].to_numpy()

    # >>> GARDE SCHÉMA SUR ROUTES (Claude, 2026-11-19)
    # Le schéma ne met plus en piste une route déjà longée par une voie cyclable OSM dessinée à part :
    # sinon la route récupère la classe de la piste voisine (ligne du schéma à 5-8 m) et compte double.
    SEPARATE_WAY_HW = {"cycleway", "path", "footway", "pedestrian", "track", "bridleway"}
    OSM_SEPARATE_DEDICATED = {"piste_cyclable", "voie_mixte", "footway_designe"}
    SCHEMA_ROAD_GUARD_HALF_WIDTH_M = 10.0
    SCHEMA_ROAD_GUARD_MAX_ANGLE_DEG = 30.0
    SCHEMA_ROAD_GUARD_MIN_SHARE = 0.5

    is_road = ~bike_edges_seg["highway"].astype(str).isin(SEPARATE_WAY_HW).to_numpy()
    osm_sep = bike_edges_seg.loc[~is_road & bike_edges_seg["infra_bike"].isin(OSM_SEPARATE_DEDICATED).to_numpy(), ["geometry"]]
    cand = np.flatnonzero(is_road & (bike_edges_seg["infra_schema"] == "piste_cyclable").to_numpy())
    ignored = np.zeros(len(bike_edges_seg), dtype=bool)
    if len(cand) and len(osm_sep):
        sub = bike_edges_seg.iloc[cand][["geometry"]].assign(_eid=np.arange(len(cand)))
        guard = extract_rectangle_majority_class(
            sub, osm_sep.assign(_voie_osm="voie_cyclable_osm"), "_voie_osm",
            SCHEMA_ROAD_GUARD_HALF_WIDTH_M, max_angle_deg=SCHEMA_ROAD_GUARD_MAX_ANGLE_DEG,
            min_share=SCHEMA_ROAD_GUARD_MIN_SHARE, crs_meter_epsg=operation_crs,
            segment_id_col="_eid", verbose=False,
        )
        ignored[cand[guard["_voie_osm"].notna().to_numpy()]] = True
    infra_schema = bike_edges_seg["infra_schema"].to_numpy(dtype=object)
    infra_schema[ignored] = None
    bike_edges_seg["infra_schema"] = infra_schema
    bike_edges_seg["schema_piste_ignoree"] = ignored
    L_c = bike_edges_seg.length.to_numpy() / 1000
    print(f"Garde schéma : {L_c[cand].sum():.1f} km de routes en « Site propre » dans le schéma, "
          f"dont {L_c[ignored].sum():.1f} km ignorés (piste OSM dessinée à part à côté)")
    # <<< FIN GARDE SCHÉMA SUR ROUTES
else:
    print(f"⚠️ Schéma cyclable absent ({SCHEMA_PATH}) : infra_cyclable = infra_bike")
    bike_edges_seg["infra_schema"] = None
    bike_edges_seg["infra_schema_share"] = 0.0
    bike_edges_seg["schema_piste_ignoree"] = False   # >>> GARDE SCHÉMA SUR ROUTES (2026-11-19)

# Union : la classe de rang le plus haut l'emporte
r_osm = bike_edges_seg["infra_bike"].map(INFRA_RANK).fillna(0)
r_sch = bike_edges_seg["infra_schema"].map(INFRA_RANK).fillna(-1)
# >>> CLASSES OSM PROTÉGÉES (Claude, 2026-11-23) : « Site propre » du schéma mélange voies vertes et pistes ;
# OSM est plus précis sur les voies mixtes et les footways désignés, le schéma ne les surclasse plus.
SCHEMA_PROTECTED_OSM = {"voie_mixte", "footway_designe"}
schema_wins = (r_sch > r_osm) & ~bike_edges_seg["infra_bike"].isin(SCHEMA_PROTECTED_OSM)
L_prot = bike_edges_seg.length[(r_sch > r_osm) & ~schema_wins].sum() / 1000
print(f"Classes OSM protégées du schéma ({', '.join(sorted(SCHEMA_PROTECTED_OSM))}) : {L_prot:.1f} km non surclassés")
# AVANT : np.where(r_sch > r_osm, ...)
bike_edges_seg["infra_cyclable"] = np.where(schema_wins, bike_edges_seg["infra_schema"], bike_edges_seg["infra_bike"])
bike_edges_seg["infra_cyclable_source"] = np.where(schema_wins, "schema", "osm")
# <<< FIN CLASSES OSM PROTÉGÉES

L = bike_edges_seg.length / 1000
print("\nLongueur par classe (km) :")
print(pd.DataFrame({"infra_bike (OSM)": L.groupby(bike_edges_seg["infra_bike"]).sum(),
                    "infra_cyclable": L.groupby(bike_edges_seg["infra_cyclable"]).sum()})
      .reindex(sorted(INFRA_RANK, key=INFRA_RANK.get, reverse=True)).round(1).fillna(0).to_string())
print(f"Arêtes reclassées par le schéma : {int((bike_edges_seg['infra_cyclable_source'] == 'schema').sum()):,}")

Schéma cyclable : 1,157 lignes retenues
infra_schema
bande_cyclable    105.7
partage            23.3
piste_cyclable    216.1
  Rectangle demi-largeur 10.0 m | morceaux retenus : 8,776 (obliques écartés : 2,911) | segments classés : 6,548 / 108,807 (part couverte >= 50%)
Garde schéma : 158.2 km de routes en « Site propre » dans le schéma, dont 104.6 km ignorés (piste OSM dessinée à part à côté)
Classes OSM protégées du schéma (footway_designe, voie_mixte) : 91.6 km non surclassés

Longueur par classe (km) :
                      infra_bike (OSM)  infra_cyclable
piste_cyclable                   262.2           319.9
bande_cyclable                   337.1           356.8
voie_mixte                       155.5           155.5
footway_designe                   64.6            64.6
contresens                        19.3            19.2
partage_bus                       32.1            27.3
accotement                        36.8            36.8
bike_pedestrian_area             310.2          

In [16]:
# 2) construire graphe OSMnx
G_walk, walk_nodes_gdf, walk_edges_gdf_graph = build_osmnx_graph_from_segmented_edges(
    walk_edges_seg,
    crs_metric=operation_crs,
    snap_precision_m=0.5,
    directed=False,
)
print("Graphe piéton construit.")

if bike_directed_graph:
    G_bike, bike_nodes_gdf, bike_edges_gdf_graph = build_directed_bike_graph_from_segmented_edges(
        bike_edges_seg,
        crs_metric=operation_crs,
        snap_precision_m=0.5,
    )
    print("Graphe vélo orienté construit.")
else:
    G_bike, bike_nodes_gdf, bike_edges_gdf_graph = build_osmnx_graph_from_segmented_edges(
        bike_edges_seg,
        crs_metric=operation_crs,
        snap_precision_m=0.5,
        directed=False,
    )
    print("Graphe vélo non-orienté construit.")

Graphe piéton construit.
Graphe vélo orienté construit.


## 8 · Enrichissements automatiques

**Les enrichissements sont appliqués automatiquement aux données exportées en section 8.**

### Nœuds OSM avec attributs

**Attributs intégrés pour nœuds :**
- **Traversées** : `crossing=*`, `traffic_signals=yes`
- **Barrières** : `barrier=bollard/cycle_barrier`
- **Apaisement** : `traffic_calming=*`
- **Transport** : `public_transport=stop_position`

### Attributs vélo intégrés

**Surface de roulage :**
- **Surface** : `surface=asphalt/gravel/paving_stones` (valeur OSM brute)

### Zones de conflit piéton-vélo

**Détection automatique :**
1. **Intersections géométriques** entre réseaux
2. **Aires partagées** : `highway=pedestrian` + `bicycle=yes`
3. **Chemins mixtes** : `highway=path` sans ségrégation

In [17]:
def enrich_nodes_with_osm_attributes(nodes_gdf: gpd.GeoDataFrame, aoi_poly, operation_crs: str) -> gpd.GeoDataFrame:
    """
    Enrichit les nœuds de graphe avec les attributs des nœuds OSM proches.
    """
    # Télécharger les nœuds OSM avec attributs intéressants
    osm_node_tags = {
        "crossing": True,
        "traffic_signals": True, 
        "barrier": True,
        "traffic_calming": True,
        "public_transport": True
    }
    
    try:
        osm_nodes = load_osm_features(aoi_poly, osm_node_tags, territory, variant="network")
        osm_nodes = gpd.GeoDataFrame(osm_nodes, geometry="geometry", crs="EPSG:4326")
        osm_nodes = osm_nodes[osm_nodes.geometry.type == "Point"].copy()
    except Exception:
        print("Aucun nœud OSM avec attributs trouvé")
        return nodes_gdf
    
    if osm_nodes.empty:
        return nodes_gdf
    
    # Convertir en CRS métrique pour calculs de distance
    nodes_metric = to_metric_crs(nodes_gdf, operation_crs)
    osm_nodes_metric = to_metric_crs(osm_nodes, operation_crs)
    
    # Associer par proximité (rayon 20m)
    snap_distance = 20.0
    nodes_enriched = nodes_gdf.copy()
    
    # Initialiser colonnes d'attributs
    for col in ["crossing", "traffic_signals", "barrier", "traffic_calming"]:
        nodes_enriched[col] = None
    
    if not osm_nodes_metric.empty:
        sindex = osm_nodes_metric.sindex
        
        for idx, node in nodes_metric.iterrows():
            candidates = list(sindex.intersection(node.geometry.buffer(snap_distance).bounds))
            if candidates:
                nearby = osm_nodes_metric.iloc[candidates]
                nearby = nearby[nearby.distance(node.geometry) <= snap_distance]
                
                if not nearby.empty:
                    # Prendre le plus proche
                    closest = nearby.loc[nearby.distance(node.geometry).idxmin()]
                    
                    # Copier attributs
                    for col in ["crossing", "traffic_signals", "barrier", "traffic_calming"]:
                        if col in closest and pd.notna(closest[col]):
                            nodes_enriched.loc[idx, col] = str(closest[col])
    
    return nodes_enriched

def enrich_bike_edges_with_surface(bike_edges: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Enrichit les arêtes vélo avec attributs de surface brute.
    """
    enriched = bike_edges.copy()
    
    # Surface brute (sans score)
    def get_surface(row) -> str:
        surface = row.get("surface", None)
        if pd.isna(surface) or surface is None:
            return None
        return str(surface)
    
    enriched["surface"] = enriched.apply(get_surface, axis=1)
    
    return enriched

def detect_pedestrian_bike_conflicts(walk_edges: gpd.GeoDataFrame, bike_edges: gpd.GeoDataFrame, 
                                   operation_crs: str) -> gpd.GeoDataFrame:
    """
    Détecte les zones de conflit potentiel entre piétons et vélos.
    """
    # Convertir en CRS métrique
    walk_metric = to_metric_crs(walk_edges, operation_crs)
    bike_metric = to_metric_crs(bike_edges, operation_crs)
    
    conflicts = []
    
    # 1. Intersections géométriques directes
    if not walk_metric.empty and not bike_metric.empty:
        walk_sindex = walk_metric.sindex
        
        for idx, bike_edge in bike_metric.iterrows():
            bike_geom = bike_edge.geometry
            if bike_geom is None or bike_geom.is_empty:
                continue
                
            # Chercher intersections avec réseau piéton
            candidates = list(walk_sindex.intersection(bike_geom.bounds))
            
            for walk_idx in candidates:
                walk_geom = walk_metric.iloc[walk_idx].geometry
                
                if bike_geom.intersects(walk_geom):
                    intersection = bike_geom.intersection(walk_geom)
                    
                    if intersection.geom_type == "Point":
                        conflicts.append({
                            "geometry": intersection,
                            "conflict_type": "intersection",
                            "walk_class": walk_metric.iloc[walk_idx].get("walk_class", "unknown"),
                            "bike_infra": bike_edge.get("infra_bike", "unknown"),
                            "severity": "high" if bike_edge.get("infra_bike") == "bike_road" else "medium"
                        })
                    elif intersection.geom_type in ["LineString", "MultiLineString"]:
                        conflicts.append({
                            "geometry": intersection.centroid,
                            "conflict_type": "shared_space",
                            "walk_class": walk_metric.iloc[walk_idx].get("walk_class", "unknown"),
                            "bike_infra": bike_edge.get("infra_bike", "unknown"),
                            "severity": "medium"
                        })
    
    # 2. Aires partagées explicites (pedestrian + bicycle=yes)
    shared_areas = bike_edges[
        (bike_edges.get("highway") == "pedestrian") & 
        (bike_edges.get("bicycle").isin(["yes", "designated"]))
    ].copy()
    
    for idx, area in shared_areas.iterrows():
        if area.geometry is not None:
            conflicts.append({
                "geometry": area.geometry.centroid,
                "conflict_type": "shared_pedestrian_area",
                "walk_class": "pedestrian_area",
                "bike_infra": "bike_pedestrian_area",
                "severity": "low"
            })
    
    if conflicts:
        conflicts_gdf = gpd.GeoDataFrame(conflicts, crs=operation_crs)
        return conflicts_gdf.to_crs("EPSG:4326")
    else:
        return gpd.GeoDataFrame(
            columns=["geometry", "conflict_type", "walk_class", "bike_infra", "severity"],
            geometry="geometry", crs="EPSG:4326"
        )

# Application des enrichissements (conditionnel)
if enable_enrichments:
    print("=== ENRICHISSEMENTS ACTIVÉS ===")
    print("Enrichissement des nœuds avec attributs OSM...")
    walk_nodes_gdf = enrich_nodes_with_osm_attributes(walk_nodes_gdf, aoi_poly, operation_crs)
    bike_nodes_gdf = enrich_nodes_with_osm_attributes(bike_nodes_gdf, aoi_poly, operation_crs)

    print("Enrichissement des arêtes vélo avec surface...")
    bike_edges_gdf_graph = enrich_bike_edges_with_surface(bike_edges_gdf_graph)

    print("Détection des conflits piéton-vélo...")
    conflict_zones = detect_pedestrian_bike_conflicts(walk_edges_gdf_graph, bike_edges_gdf_graph, operation_crs)
    print(f"Conflits détectés: {len(conflict_zones)}")
    
    # Export des zones de conflit si détectées
    if not conflict_zones.empty and export_edges:
        export_gdf(conflict_zones, "pedestrian_bike_conflicts", export_formats, output_dir, export_crs)
        print(f"Zones de conflit exportées: {len(conflict_zones)}")
    
    print("\n=== ENRICHISSEMENTS INTÉGRÉS ===")
    print("Les nœuds et arêtes exportés incluent maintenant:")
    print("- Nœuds: attributs OSM (crossing, traffic_signals, barrier, traffic_calming)")
    print("- Arêtes vélo: surface OSM brute")
    print(f"- Zones de conflit: {len(conflict_zones)} détectées")
else:
    print("=== ENRICHISSEMENTS DÉSACTIVÉS ===")
    print("Export des données de base uniquement (sans attributs OSM)")
    conflict_zones = gpd.GeoDataFrame(
        columns=["geometry", "conflict_type", "walk_class", "bike_infra", "severity"],
        geometry="geometry", crs="EPSG:4326"
    )

=== ENRICHISSEMENTS ACTIVÉS ===
Enrichissement des nœuds avec attributs OSM...
Enrichissement des arêtes vélo avec surface...
Détection des conflits piéton-vélo...
Conflits détectés: 954039
Zones de conflit exportées: 954039

=== ENRICHISSEMENTS INTÉGRÉS ===
Les nœuds et arêtes exportés incluent maintenant:
- Nœuds: attributs OSM (crossing, traffic_signals, barrier, traffic_calming)
- Arêtes vélo: surface OSM brute
- Zones de conflit: 954039 détectées


## 10 · Export 

**GraphML** : Format NetworkX pour analyses de graphes avancées

**Formats de sortie configurables :**

- **Parquet** : Format optimal (compression, typage, performance)
- **GeoJSON** : Interopérabilité web/JS
- **GPKG** : SIG desktop (QGIS, ArcGIS)

**CRS final** : Export en EPSG:4326 (standard géographique)

**Couches produites  :**
- **Nodes graph** : Nœuds de graphe + attributs OSM (crossing, traffic_signals, barrier, traffic_calming)
- **Edges graph** : Arêtes de graphe + surface OSM pour vélo
- **Crossing nodes** : Points de traversée OSM
- **Conflict zones** : Zones de conflit piéton-vélo (si détectées)

In [18]:
if export_edges:
    export_gdf(walk_edges_seg, "walk_edges_segmented", export_formats, output_dir, export_crs)
    export_gdf(walk_edges_gdf_graph, "walk_edges_graph", export_formats, output_dir, export_crs)
    export_gdf(bike_edges_seg, "bike_edges_segmented", export_formats, output_dir, export_crs)
    export_gdf(bike_edges_gdf_graph, "bike_edges_graph", export_formats, output_dir, export_crs)

if export_nodes:
    export_gdf(walk_nodes_gdf, "walk_nodes_graph", export_formats, output_dir, export_crs)
    export_gdf(bike_nodes_gdf, "bike_nodes_graph", export_formats, output_dir, export_crs)
    if len(nodes_cross) > 0:
        export_gdf(nodes_cross, "walk_crossing_nodes", export_formats, output_dir, export_crs)

if export_graphml:
    ox.save_graphml(G_walk, filepath=str(output_dir / "walk_graph.graphml"))

In [19]:
def ids_res(df):
    m = df["highway"].eq("residential")
    return set(df.loc[m, "osm_id"].dropna())

stages = {
    "1_telechargement": ways_raw,
    "3_apres_proxy": walk_edges,
    "6-7_redondance_dedup": walk_edges_p,
    "8_segmente": walk_edges_seg,
    "9_graphe": walk_edges_gdf_graph,
}
prev = None
for name, df in stages.items():
    s = ids_res(df)
    perte = "" if prev is None else f"  (-{len(prev - s)})"
    print(f"{name:24s}{len(s):6d} ways residential{perte}")
    prev = s

rm = removed_sidewalk_proxy_duplicates
export_gdf(rm[rm["highway"].eq("residential")],
           "debug_residential_removed_redundancy", ["gpkg"], output_dir, export_crs)

1_telechargement         17019 ways residential
3_apres_proxy            17003 ways residential  (-16)
6-7_redondance_dedup     14789 ways residential  (-2214)
8_segmente               14788 ways residential  (-1)
9_graphe                 14764 ways residential  (-24)
